<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/OpenVoice-V2_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# OpenVoice V2 — Instant Multilingual Voice Cloning & Conversion

OpenVoice is the open-source voice-cloning / voice-conversion model from [MIT + MyShell](https://research.myshell.ai/open-voice). It clones the **tone color** (timbre / accent / rhythm / emotion) of a short reference clip and applies it to **any source audio**, with native multi-lingual support (English, Spanish, French, Chinese, Japanese, Korean) via MeloTTS as the base speaker. **MIT licensed — free for commercial use.**

> "OpenVoice can accurately clone the reference tone color and generate speech in multiple languages and accents." — [arXiv 2312.01479](https://arxiv.org/abs/2312.01479)

**Upstream**: [myshell-ai/OpenVoice](https://github.com/myshell-ai/OpenVoice) · [myshell-ai/OpenVoiceV2](https://huggingface.co/myshell-ai/OpenVoiceV2) · [arXiv 2312.01479](https://arxiv.org/abs/2312.01479) · [MIT](https://github.com/myshell-ai/OpenVoice/blob/main/LICENSE)

## What this notebook does

- **Voice conversion** — change the *timbre* of an audio clip to match a short reference voice (5–30 s), keeping the *content / prosody* of the source
- **Text-to-speech with cloned voice** — type text in EN/ES/FR/ZH/JA/KR, get it spoken in the reference voice
- **Tone color control** — adjust emotion, accent, rhythm, and pitch via the `tau` slider
- **Watermarking** — every output is watermarked with an embedded message (default `@MyShell`) via `wavmark`
- **Six tabs**: Convert, TTS+Convert, Style Controls, Batch, VRAM, Help
- **Two model versions in one notebook** — V1 (simpler, EN/ZH only) and V2 (multilingual, better quality)

## What this notebook does NOT do

- **Real-time streaming** voice conversion (Seed-VC does this but is GPL-3.0, see Seed-VC roadmap)
- **Training a new base speaker** — only zero-shot cloning from reference audio
- **Transcription / ASR** — the source audio's words are preserved as-is

> GPU: T4 works but **L4+ (22+ GB) recommended** for larger reference audio and batch operations.

## How to use

1. Run **STEP 1** (installs OpenVoice + MeloTTS + Japanese dictionary, ~5 min first time)
2. Run **STEP 2** (downloads V1+V2 checkpoints, ~300 MB)
3. Run **STEP 3** (core functions — fast)
4. Run **STEP 4** (launches the Gradio UI)
5. For scripting, use **Step 6** (quick test) or **Step 7** (batch)


In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Installs the `openvoice` package (with bundled MeloTTS for V2 multilingual), `unidic` for Japanese G2P, `ffmpeg` (system), and `wavmark` for watermarking. ~5 min first time.

import os, sys, subprocess, time, re

# GPU auto-detect
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[GPU] torch {torch.__version__} · device = {DEVICE}', flush=True)
if DEVICE == 'cuda':
    try:
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'[GPU] {name} ({vram:.1f} GB VRAM)', flush=True)
    except Exception as e:
        print(f'[GPU] (could not read device name: {e})', flush=True)

# Drive cache setup — OpenVoice checkpoints are ~300 MB total for V1+V2
DRIVE = '/content/drive/MyDrive/AEI_TTS_Cache/huggingface'
os.environ.setdefault('HF_HOME', DRIVE)
os.makedirs(DRIVE, exist_ok=True)
print(f'[Cache] HF_HOME = {DRIVE}', flush=True)

# Output dirs
os.makedirs('/content/voice_out', exist_ok=True)
os.makedirs('/content/ref_audio', exist_ok=True)
print('[Setup] /content/voice_out and /content/ref_audio ready.', flush=True)

# System package: ffmpeg (required by pydub)
print('\n[apt] Installing ffmpeg...', flush=True)
subprocess.run(['apt-get', '-qq', '-y', 'install', 'ffmpeg'],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
import shutil
print(f'[apt] ffmpeg = {shutil.which("ffmpeg")}', flush=True)

# ── Step 1a: Install OpenVoice + MeloTTS from git (--no-deps skips version-pinned setup.py) ──
print('\n[pip] Installing OpenVoice + MeloTTS (git) — ~3-5 min...', flush=True)
t0 = time.time()
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'cython',
     'git+https://github.com/myshell-ai/OpenVoice.git',
     'git+https://github.com/myshell-ai/MeloTTS.git'], check=True)

# ── Step 1b: Discover & install all runtime deps from the repos' setup.py/requirements.txt ──
# Install all runtime deps explicitly (version pins relaxed for Colab torch 2.11)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'eng_to_ipa', 'inflect', 'unidecode', 'pypinyin', 'cn2an', 'jieba', 'langid',
     'wavmark', 'pydub', 'librosa', 'faster-whisper', 'whisper-timestamped',
     'txtsplit', 'cached_path', 'num2words', 'unidic_lite', 'unidic',
     'mecab-python3', 'pykakasi', 'fugashi', 'g2p_en', 'anyascii', 'jamo',
     'g2pkk', 'gruut', 'loguru', 'tensorboard', 'scipy', 'safetensors',
     'tqdm', 'hf_transfer', 'gradio>=4.0'],
    check=True,
)
print(f'[pip] Done in {time.time()-t0:.1f}s', flush=True)

# ── Step 1c: Japanese dictionary ──
print('\n[unidic] Downloading Japanese dictionary...', flush=True)
try:
    import unidic
    unidic.DICDIR = '/content/unidic_data'
    unidic.download()
    print('[unidic] Done', flush=True)
except Exception as e:
    print(f'[unidic] skipped: {e} (only needed for JP language)', flush=True)

# ── Sanity-check the imports ──
print('\n[check] Verifying imports...', flush=True)
try:
    import openvoice
    from openvoice.api import ToneColorConverter
    from openvoice import se_extractor
    print(f'[openvoice] imported OK', flush=True)
except Exception as e:
    print(f'[openvoice] FAILED: {e}', flush=True)
    raise

py_ver = sys.version_info
print(f'[Python] {py_ver.major}.{py_ver.minor}.{py_ver.micro}', flush=True)
print('\n[Done] STEP 1 complete.', flush=True)


In [ ]:
# @title STEP 2 — Pre-cache Model Checkpoints
# @markdown Downloads the OpenVoice V1 + V2 checkpoints from Hugging Face (MyShell moved them from S3).

# @markdown **V1 size: ~500 MB (with base speakers) · V2 size: ~130 MB · MeloTTS models: ~150 MB each on demand.**

import os, shutil, time
from huggingface_hub import snapshot_download

CACHE_ROOT = '/content/openvoice_models'
os.makedirs(CACHE_ROOT, exist_ok=True)

# MyShell moved checkpoints from S3 to Hugging Face.
# V1: myshell-ai/OpenVoice (includes base_speakers + converter)
# V2: myshell-ai/OpenVoiceV2 (includes base_speakers/ses/ + converter)
V1_DIR = os.path.join(CACHE_ROOT, 'checkpoints_v1')
V2_DIR = os.path.join(CACHE_ROOT, 'checkpoints_v2')

for label, repo_id, target in [
    ('V1', 'myshell-ai/OpenVoice', V1_DIR),
    ('V2', 'myshell-ai/OpenVoiceV2', V2_DIR),
]:
    if os.path.isdir(target) and os.listdir(target):
        print(f'[{label}] Already cached at {target}', flush=True)
        continue
    print(f'[{label}] Downloading {repo_id} ...', flush=True)
    t0 = time.time()
    try:
        tmp = snapshot_download(repo_id)
        if os.path.exists(target):
            shutil.rmtree(target)
        shutil.copytree(tmp, target, symlinks=True)
        size_mb = sum(os.path.getsize(os.path.join(dp,f)) for dp,_,fs in os.walk(target) for f in fs)/1e6
        print(f'[{label}] {size_mb:.1f} MB in {time.time()-t0:.1f}s', flush=True)
    except Exception as e:
        print(f'[{label}] FAILED: {e}', flush=True)

print('\n[Done] STEP 2 complete.', flush=True)


In [ ]:
# @title STEP 3 — Core Functions (lazy model loading)
# @markdown Defines the inference wrappers. The V1/V2 tone-color converters and MeloTTS base speakers are loaded on first use, not at import time. This cell is fast.

import os, time, gc
import numpy as np
import torch
import soundfile as sf
from typing import Optional, Tuple

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAMPLE_RATE = 22050  # OpenVoice output rate (matches VITS internal)

# ── Language support table ────────────────────────────────────────
# (melo_lang_code, display label, base_se_filename_stem)
MELO_LANGS = [('EN_NEWEST', 'English 🇺🇸 (newest)', 'en_newest'), ('EN', 'English 🇺🇸 (default)', 'en-default'), ('ES', 'Spanish 🇪🇸', 'es'), ('FR', 'French 🇫🇷', 'fr'), ('ZH', 'Chinese 🇨🇳 (mixed EN)', 'zh'), ('JP', 'Japanese 🇯🇵', 'jp'), ('KR', 'Korean 🇰🇷', 'kr')]

CACHE_ROOT = '/content/openvoice_models'
V1_CKPT_DIR = os.path.join(CACHE_ROOT, 'checkpoints_v1')
V2_CKPT_DIR = os.path.join(CACHE_ROOT, 'checkpoints_v2')

# ── Lazy converter cache (V1 and V2 separately) ─────────────────
CONVERTERS = {}  # version -> ToneColorConverter


def get_converter(version: str = 'v2'):
    """Lazily build a ToneColorConverter for the given OpenVoice version.
    version: 'v1' (simpler, EN/ZH only) | 'v2' (multilingual via MeloTTS)
    """
    version = version.lower()
    if version in CONVERTERS and CONVERTERS[version] is not None:
        return CONVERTERS[version]
    from openvoice.api import ToneColorConverter
    if version == 'v1':
        ckpt_dir = V1_CKPT_DIR
    elif version == 'v2':
        ckpt_dir = V2_CKPT_DIR
    else:
        raise ValueError(f"version must be 'v1' or 'v2', got {version!r}")
    print(f'[Converter] {version} from {ckpt_dir}...', flush=True)
    t0 = time.time()
    conv = ToneColorConverter(f'{ckpt_dir}/converter/config.json', device=DEVICE)
    conv.load_ckpt(f'{ckpt_dir}/converter/checkpoint.pth')
    CONVERTERS[version] = conv
    print(f'[Converter] {version} ready in {time.time()-t0:.1f}s', flush=True)
    return conv


def free_converters():
    global CONVERTERS
    CONVERTERS.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('[Free] All converters released', flush=True)


# ── MeloTTS base speaker cache ───────────────────────────────────
BASE_SPEAKERS = {}  # (version, lang_code) -> loaded speaker


def get_base_speaker(version: str, lang_code: str):
    key = (version, lang_code)
    if key in BASE_SPEAKERS:
        return BASE_SPEAKERS[key]
    from melo.api import TTS
    print(f'[MeloTTS] Loading base speaker: {lang_code}...', flush=True)
    t0 = time.time()
    # MeloTTS downloads language models on first use
    speaker = TTS(language=lang_code, device=DEVICE)
    BASE_SPEAKERS[key] = speaker
    print(f'[MeloTTS] {lang_code} ready in {time.time()-t0:.1f}s', flush=True)
    return speaker


# ── Save helper ──────────────────────────────────────────────────
def save_wav(audio: np.ndarray, sr: int, name: str) -> str:
    path = os.path.join('/content/voice_out', name)
    sf.write(path, audio, sr, subtype='PCM_16')
    return path


# ── Voice conversion ─────────────────────────────────────────────
def convert_voice(source_audio, ref_audio, version='v2', tau=0.3, watermark='@MyShell', progress=None):
    """Convert source audio to the timbre of the reference voice.
    source_audio: filepath, (sr, np.ndarray) tuple, or np.ndarray (inferred sr=22050)
    ref_audio: same formats
    Returns (audio_np, sample_rate)"""
    from openvoice import se_extractor

    # Materialize inputs
    def _load(thing, label):
        if isinstance(thing, str):
            data, sr = sf.read(thing)
            return np.asarray(data, dtype=np.float32), sr
        if isinstance(thing, (list, tuple)) and len(thing) == 2:
            a, b = thing
            if isinstance(b, np.ndarray):
                return np.asarray(b, dtype=np.float32), int(a)
            return np.asarray(a, dtype=np.float32), int(b)
        if isinstance(thing, np.ndarray):
            return np.asarray(thing, dtype=np.float32), SAMPLE_RATE
        raise ValueError(f'{label}: expected filepath, (sr, ndarray), or ndarray, got {type(thing)}')

    src, src_sr = _load(source_audio, 'source_audio')
    ref, ref_sr = _load(ref_audio, 'ref_audio')

    if progress:
        progress(0.1, 'Loading converters...')
    conv = get_converter(version)
    if progress:
        progress(0.3, 'Extracting tone color...')
    tone = se_extractor.get_se(ref, conv, target_dir='/content')
    if progress:
        progress(0.6, 'Converting voice...')
    out = conv.convert(src, tone, src_sr, tau=tau, message=watermark)
    if progress:
        progress(1.0, 'Done')
    return np.asarray(out, dtype=np.float32), SAMPLE_RATE


# ── TTS with cloned voice (MeloTTS as base) ─────────────────────
def tts_with_voice(text, ref_audio, lang='EN_NEWEST', speed=1.0, version='v2', tau=0.3, watermark='@MyShell', progress=None):
    """Generate speech with MeloTTS then convert to reference voice timbre."""
    from openvoice import se_extractor

    # Load reference
    if isinstance(ref_audio, str):
        ref, ref_sr = sf.read(ref_audio)
        ref = np.asarray(ref, dtype=np.float32)
    elif isinstance(ref_audio, (list, tuple)) and len(ref_audio) == 2:
        a, b = ref_audio
        ref = np.asarray(b if isinstance(b, np.ndarray) else a, dtype=np.float32)
    else:
        raise ValueError(f'ref_audio: expected filepath or (sr, ndarray), got {type(ref_audio)}')

    if progress:
        progress(0.05, 'Loading MeloTTS...')
    speaker = get_base_speaker(version, lang)
    if progress:
        progress(0.15, 'Generating base speech...')
    # MeloTTS returns filepath
    base_wav = speaker.tts_to_file(text, speaker.hps.data.spk_id if hasattr(speaker, 'hps') else 0,
                                     speed=float(speed), quiet=True)
    if progress:
        progress(0.4, 'Loading converters...')
    conv = get_converter(version)
    if progress:
        progress(0.5, 'Extracting tone color...')
    tone = se_extractor.get_se(ref, conv, target_dir='/content')
    if progress:
        progress(0.7, 'Converting voice...')
    out = conv.convert(base_wav, tone, SAMPLE_RATE, tau=tau, message=watermark)
    if progress:
        progress(1.0, 'Done')
    return np.asarray(out, dtype=np.float32), SAMPLE_RATE


print('[Core] Ready. Use STEP 4 for the UI, Step 6 for quick test, Step 7 for batch.')


In [ ]:
# @title STEP 4 — Gradio UI
# @markdown Interactive web app with six tabs: **Convert** (audio in + ref audio in → audio out), **TTS+Convert** (text + ref audio → audio out), **Style Controls** (emotion/accent/tau), **Batch** (zip of conversions), **VRAM** (free loaded models), **Help** (multilingual notes, citation).

import os, sys, time, json, io, zipfile
import numpy as np
import soundfile as sf
import gradio as gr

# ── Colab / nest_asyncio so Gradio can run inside the notebook loop ──
try:
    import nest_asyncio
    nest_asyncio.apply()
except Exception:
    pass
try:
    from IPython.display import clear_output as _clear
    _clear()
except Exception:
    pass


def _normalize_audio(audio):
    """Gradio can pass filepath or (sr, ndarray). Return (sr, ndarray) or None."""
    if audio is None:
        return None
    if isinstance(audio, (list, tuple)):
        if len(audio) == 2 and isinstance(audio[1], np.ndarray):
            return int(audio[0]), audio[1]
        return None
    if isinstance(audio, str):
        data, sr = sf.read(audio)
        return int(sr), np.asarray(data, dtype=np.float32)
    return None


# ── Tab 1: Voice Convert ────────────────────────────────────────
def ui_convert(src_audio, ref_audio, version, tau, watermark, progress=gr.Progress()):
    if not ref_audio:
        raise gr.Error('Upload a reference audio (5-30s of clear speech).')
    if not src_audio:
        raise gr.Error('Upload a source audio to convert.')
    try:
        audio, sr = convert_voice(src_audio, ref_audio, version, tau, watermark, progress)
    except Exception as e:
        raise gr.Error(f'Conversion failed: {e}')
    fname = f'convert_{int(time.time())}.wav'
    path = save_wav(audio, sr, fname)
    meta = (
        f"version = {version} · tau = {tau:.2f} · watermark = '{watermark}'\n"
        f"duration = {len(audio)/sr:.2f}s · sample_rate = {sr} Hz\n"
        f"src = {os.path.basename(src_audio) if isinstance(src_audio, str) else 'upload'}\n"
        f"ref = {os.path.basename(ref_audio) if isinstance(ref_audio, str) else 'upload'}"
    )
    return (sr, audio), path, meta


# ── Tab 2: TTS + Clone ──────────────────────────────────────────
def ui_tts_clone(text, ref_audio, lang, speed, version, tau, watermark, progress=gr.Progress()):
    if not text or not text.strip():
        raise gr.Error('Type some text first.')
    if not ref_audio:
        raise gr.Error('Upload a reference audio (5-30s of clear speech).')
    if version == 'v1' and lang not in ('EN', 'ZH'):
        raise gr.Error('V1 only supports English (EN) and Chinese (ZH). Switch to V2 for multilingual.')
    try:
        lang_code = dict(MELO_LANGS)[lang]
        audio, sr = tts_with_voice(text, ref_audio, lang_code, speed, version, tau, watermark, progress)
    except Exception as e:
        raise gr.Error(f'TTS failed: {e}')
    fname = f'tts_{int(time.time())}.wav'
    path = save_wav(audio, sr, fname)
    meta = (
        f"version = {version} · lang = {lang} · speed = {speed}\n"
        f"duration = {len(audio)/sr:.2f}s · sample_rate = {sr} Hz"
    )
    return (sr, audio), path, meta


# ── Tab 3: Style Controls ───────────────────────────────────────
def ui_style_demo(ref_audio, version, tau, watermark, progress=gr.Progress()):
    """Quick demo: convert a built-in tone to the reference voice at different tau values."""
    if not ref_audio:
        raise gr.Error('Upload a reference audio first.')
    tone, log = get_tone(ref_audio, version, progress)
    if tone is None:
        raise gr.Error(f'Tone extraction failed: {log}')
    return f'✓ Tone extracted ({tone.shape[0]} dims)\n  tau={tau:.2f} · watermark={watermark!r}'


def get_tone(ref_audio, version, progress=None):
    from openvoice import se_extractor
    if isinstance(ref_audio, str):
        ref, _ = sf.read(ref_audio)
        ref = np.asarray(ref, dtype=np.float32)
    elif isinstance(ref_audio, (list, tuple)) and len(ref_audio) == 2:
        a, b = ref_audio
        ref = np.asarray(b if isinstance(b, np.ndarray) else a, dtype=np.float32)
    else:
        return None, 'Invalid reference audio format'
    try:
        conv = get_converter(version)
        tone = se_extractor.get_se(ref, conv, target_dir='/content')
        return tone, f'OK ({tone.shape[0]} dims)'
    except Exception as e:
        return None, str(e)


# ── Tab 4: Batch ─────────────────────────────────────────────────
def ui_batch(input_dir, ref_audio, version, tau, watermark, progress=gr.Progress()):
    if not ref_audio or not input_dir:
        raise gr.Error('Provide both an input directory and reference audio.')
    if not os.path.isdir(input_dir):
        raise gr.Error(f'Input dir not found: {input_dir}')
    exts = ('.wav', '.mp3', '.flac', '.ogg', '.m4a', '.opus')
    files = sorted(f for f in os.listdir(input_dir) if f.lower().endswith(exts))
    if not files:
        raise gr.Error('No audio files found in the input directory.')
    out_dir = '/content/voice_out/batch'
    os.makedirs(out_dir, exist_ok=True)
    zip_buf = io.BytesIO()
    log = []
    with zipfile.ZipFile(zip_buf, 'w', zipfile.ZIP_DEFLATED) as zf:
        for i, fname in enumerate(files):
            try:
                audio, sr = convert_voice(
                    os.path.join(input_dir, fname), ref_audio, version, tau, watermark
                )
                out_name = f'{i:03d}__{os.path.splitext(fname)[0]}.wav'
                wav_path = os.path.join(out_dir, out_name)
                sf.write(wav_path, audio, sr, subtype='PCM_16')
                with open(wav_path, 'rb') as f:
                    zf.writestr(out_name, f.read())
                log.append(f'[{i+1:03d}/{len(files)}] ok · {fname} → {len(audio)/sr:.2f}s')
            except Exception as e:
                log.append(f'[{i+1:03d}/{len(files)}] FAILED · {fname}: {e}')
            progress((i+1)/max(len(files), 1), f'{i+1}/{len(files)}')
    zip_path = '/content/voice_out/batch.zip'
    with open(zip_path, 'wb') as f:
        f.write(zip_buf.getvalue())
    return zip_path, '\n'.join(log)


# ── Tab 5: VRAM ──────────────────────────────────────────────────
def ui_vram_info():
    loaded = [v for v in CONVERTERS if CONVERTERS[v] is not None]
    msg = f'Loaded converters: {loaded or "none"}\n'
    if torch.cuda.is_available():
        free, tot = torch.cuda.mem_get_info(0)
        msg += f'GPU memory: {free/1e9:.1f} / {tot/1e9:.1f} GB free'
    else:
        msg += 'CPU mode'
    return msg


def ui_free_all():
    free_converters()
    return 'Freed all converters.'


# ── Build the UI ────────────────────────────────────────────────
with gr.Blocks(title='OpenVoice V2', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# OpenVoice V2 — Instant Voice Cloning & Conversion\n'
                'MIT licensed · 6 languages · V1 + V2 support')

    with gr.Tabs():
        # ── Tab 1: Convert ──
        with gr.Tab('Convert'):
            gr.Markdown('**Voice conversion** — change the timbre of an audio clip to match a reference voice.')
            with gr.Row():
                with gr.Column():
                    c_src = gr.Audio(label='Source audio', type='filepath'
                                      label='The audio whose voice will be changed.')
                    c_ref = gr.Audio(label='Reference voice (5-30s of clear speech)',
                                      type='filepath'
                                      label='The voice to clone.')
                    c_version = gr.Dropdown(['v1', 'v2'], value='v2', label='OpenVoice version',
                                             info='V2 = multilingual (recommended), V1 = EN/ZH only.')
                    c_tau = gr.Slider(0.1, 0.7, value=0.3, step=0.05, label='Tau (timbre similarity)',
                                       info='Higher = more similar to reference, Lower = more varied.')
                    c_wm = gr.Textbox(label='Watermark', value='@MyShell',
                                       info='Embedded audio watermark. Empty to disable.')
                    c_btn = gr.Button('🎤 Convert voice', variant='primary')
                with gr.Column():
                    c_audio = gr.Audio(label='Output', type='numpy')
                    c_file = gr.File(label='Download .wav')
                    c_info = gr.Textbox(label='Run info', lines=4, interactive=False)
            c_btn.click(ui_convert, [c_src, c_ref, c_version, c_tau, c_wm], [c_audio, c_file, c_info])

        # ── Tab 2: TTS + Clone ──
        with gr.Tab('TTS + Clone'):
            gr.Markdown('**Text-to-speech with cloned voice** — type text, get it spoken in the reference voice.')
            with gr.Row():
                with gr.Column():
                    t_text = gr.Textbox(label='Text', lines=4, value='Hello! This is my cloned voice.',
                                         info='What the cloned voice will say.')
                    t_ref = gr.Audio(label='Reference voice (5-30s)', type='filepath')
                    t_lang = gr.Dropdown(
                        [l[1] for l in MELO_LANGS], value='English 🇺🇸 (newest)',
                        label='Base speaker language',
                        info='The language for the base MeloTTS speech.')
                    t_speed = gr.Slider(0.5, 2.0, value=1.0, step=0.1, label='Speed')
                    t_version = gr.Dropdown(['v1', 'v2'], value='v2', label='OpenVoice version')
                    t_tau = gr.Slider(0.1, 0.7, value=0.3, step=0.05, label='Tau')
                    t_wm = gr.Textbox(label='Watermark', value='@MyShell')
                    t_btn = gr.Button('🗣️ TTS with cloned voice', variant='primary')
                with gr.Column():
                    t_audio = gr.Audio(label='Output', type='numpy')
                    t_file = gr.File(label='Download .wav')
                    t_info = gr.Textbox(label='Run info', lines=4, interactive=False)
            t_btn.click(ui_tts_clone, [t_text, t_ref, t_lang, t_speed, t_version, t_tau, t_wm],
                         [t_audio, t_file, t_info])

        # ── Tab 3: Style Controls ──
        with gr.Tab('Style'):
            gr.Markdown('**Tone color controls** — extract the tone color embedding and preview it.')
            with gr.Row():
                with gr.Column():
                    s_ref = gr.Audio(label='Reference voice', type='filepath')
                    s_version = gr.Dropdown(['v1', 'v2'], value='v2', label='Version')
                    s_tau = gr.Slider(0.1, 0.7, value=0.3, step=0.05, label='Tau')
                    s_wm = gr.Textbox(label='Watermark', value='@MyShell')
                    s_btn = gr.Button('🔍 Extract tone', variant='secondary')
                with gr.Column():
                    s_info = gr.Textbox(label='Tone info', lines=3, interactive=False)
            s_btn.click(ui_style_demo, [s_ref, s_version, s_tau, s_wm], [s_info])

        # ── Tab 4: Batch ──
        with gr.Tab('Batch'):
            gr.Markdown('**Batch conversion** — convert every audio file in a directory.')
            with gr.Row():
                with gr.Column():
                    b_dir = gr.Textbox(label='Input directory', value='/content/voice_out/batch_in',
                                        info='Folder with .wav/.mp3/.flac files.')
                    b_ref = gr.Audio(label='Reference voice', type='filepath')
                    b_version = gr.Dropdown(['v1', 'v2'], value='v2', label='Version')
                    b_tau = gr.Slider(0.1, 0.7, value=0.3, step=0.05, label='Tau')
                    b_wm = gr.Textbox(label='Watermark', value='@MyShell')
                    b_btn = gr.Button('📦 Run batch', variant='primary')
                with gr.Column():
                    b_zip = gr.File(label='Download .zip')
                    b_log = gr.Textbox(label='Log', lines=10, interactive=False)
            b_btn.click(ui_batch, [b_dir, b_ref, b_version, b_tau, b_wm], [b_zip, b_log])

        # ── Tab 5: VRAM ──
        with gr.Tab('VRAM'):
            v_log = gr.Textbox(label='GPU status', lines=3, interactive=False)
            v_btn = gr.Button('🧹 Free loaded models', variant='secondary')
            v_btn.click(ui_free_all, outputs=v_log)
            demo.load(ui_vram_info, outputs=v_log)

        # ── Tab 6: Help ──
        with gr.Tab('Help'):
            gr.Markdown(
                """## Supported languages

| Language | MeloTTS code | V1? | V2? |
|---|---|---|---|
| English (US) | EN, EN_NEWEST | ✓ | ✓ |
| Spanish | ES | ✗ | ✓ |
| French | FR | ✗ | ✓ |
| Chinese | ZH | ✓ | ✓ |
| Japanese | JP | ✗ | ✓ |
| Korean | KR | ✗ | ✓ |

## Style controls

- **Tau** (0.1 – 0.7, default 0.3): Tone color similarity. Higher = closer to reference timbre, lower = more variation
- **Watermark**: Every output audio is watermarked with the given text (use empty string to disable). Uses `wavmark` inaudible watermarking.

## License

[MIT License](https://github.com/myshell-ai/OpenVoice/blob/main/LICENSE) — free for commercial use.

## Citation

```bibtex
@article{qin2023openvoice,
  title   = {OpenVoice: Versatile Instant Voice Cloning},
  author  = {Qin, Zengyi and others},
  journal = {arXiv preprint arXiv:2312.01479},
  year    = {2023}
}
```
"""
            )

demo.queue(max_size=8, concurrency_limit=2).launch(
    share=True, show_error=True, server_name='0.0.0.0', server_port=7860,
)


In [ ]:
# @title Step 5 — Keep Alive
# @markdown Prevents Colab from disconnecting during long sessions. Run anytime after launching the UI.

import threading, time
def _keep():
    while True:
        time.sleep(60)
        try:
            import requests
            requests.get('https://www.google.com', timeout=5)
        except Exception:
            pass
threading.Thread(target=_keep, daemon=True).start()
print('[Keep-Alive] Running. Stop cell to disable.')


In [ ]:
# @title Step 6 — Quick Test (one-off conversion, no UI)
# @markdown Run a single voice-conversion call from the notebook. Useful for scripting or quick checks.

VERSION = "v2"  # @param ["v1", "v2"]
TAU = 0.3  # @param {type:"slider", min:0.1, max:0.7, step:0.05}
SRC_AUDIO = ""  # @param {type:"string"}
REF_AUDIO = ""  # @param {type:"string"}
WATERMARK = "@MyShell"  # @param {type:"string"}

import os, time
if not SRC_AUDIO or not os.path.exists(SRC_AUDIO):
    raise SystemExit('SRC_AUDIO is required (path to a .wav/.mp3/.flac file).')
if not REF_AUDIO or not os.path.exists(REF_AUDIO):
    raise SystemExit('REF_AUDIO is required (path to a .wav/.mp3/.flac file).')

print(f'[Test] version={VERSION} tau={TAU:.2f} src={os.path.basename(SRC_AUDIO)} ref={os.path.basename(REF_AUDIO)}')
t0 = time.time()
audio, sr = convert_voice(SRC_AUDIO, REF_AUDIO, VERSION, TAU, WATERMARK)
elapsed = time.time() - t0
path = save_wav(audio, sr, f'quicktest_{int(time.time())}.wav')
print(f'\n[Done] {path}')
print(f'[Done] {len(audio)} samples · {len(audio)/sr:.2f}s @ {sr} Hz · {elapsed:.1f}s wall')

from IPython.display import Audio, display
display(Audio(path, autoplay=False))


In [ ]:
# @title Step 7 — Batch Conversion
# @markdown Convert every audio file in a directory using the same reference voice. Each input becomes one output .wav, plus a zip of all results.

VERSION = "v2"  # @param ["v1", "v2"]
TAU = 0.3  # @param {type:"slider", min:0.1, max:0.7, step:0.05}
REF_AUDIO = ""  # @param {type:"string"}
SRC_DIR = "/content/voice_out/batch_in"  # @param {type:"string"}
OUT_DIR = "/content/voice_out/batch"  # @param {type:"string"}
WATERMARK = "@MyShell"  # @param {type:"string"}

import os, time
if not REF_AUDIO or not os.path.exists(REF_AUDIO):
    raise SystemExit('REF_AUDIO is required (path to a .wav/.mp3/.flac file).')
if not SRC_DIR or not os.path.isdir(SRC_DIR):
    raise SystemExit(f'SRC_DIR is required and must be a directory (got {SRC_DIR!r}).')

exts = ('.wav', '.mp3', '.flac', '.ogg', '.m4a', '.opus')
files = sorted(f for f in os.listdir(SRC_DIR) if f.lower().endswith(exts))
os.makedirs(OUT_DIR, exist_ok=True)
print(f'[Batch] {len(files)} files in {SRC_DIR} · version={VERSION} tau={TAU:.2f}')

t_start = time.time()
for i, fname in enumerate(files):
    src_path = os.path.join(SRC_DIR, fname)
    label = f"[{i+1:03d}/{len(files)}]"
    print(f'{label} {fname}', flush=True)
    t0 = time.time()
    try:
        audio, sr = convert_voice(src_path, REF_AUDIO, VERSION, TAU, WATERMARK)
        out_name = f'{i:03d}__{os.path.splitext(fname)[0]}.wav'
        out_path = os.path.join(OUT_DIR, out_name)
        sf.write(out_path, audio, sr, subtype='PCM_16')
        print(f'  ✓ {len(audio)/sr:.2f}s · {time.time()-t0:.1f}s · {out_path}', flush=True)
    except Exception as e:
        print(f'  ✗ EXCEPTION: {e}', flush=True)
        continue

elapsed = time.time() - t_start
print(f'\n[Done] {len(files)} files in {OUT_DIR} · total wall time {elapsed:.1f}s')
